[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ronniewillaert/SPM-Textbook-Python/blob/main/notebooks/part-I-introduction/ch02/AFM_Force_Distance_LJ_Hertz_Adhesion.ipynb)

# 📘 AFM Force–Distance Simulation Notebook
**Lennard–Jones + Contact Mechanics + Adhesion + Hysteresis**

*Accompanying notebook for Chapter 2 — Scanning Probe Microscopy textbook*

In [ ]:
# ── Google Colab setup (run this cell first) ────────────────────────
import sys
if "google.colab" in sys.modules:
    !pip install ipywidgets -q
    from google.colab import output
    output.enable_custom_widget_manager()
    print("✅ Colab environment ready — widgets enabled.")
else:
    print("ℹ️  Running locally — ensure ipywidgets is installed.")

🔷 1. Introduction
In Atomic Force Microscopy (AFM), force–distance curves reflect the combined effects of:
Interatomic attraction (Lennard–Jones interaction)
Cantilever mechanics and stability
Elastic contact (Hertz model)
Adhesion (DMT/JKR-inspired models)
Energy dissipation (hysteresis)
In this notebook, we construct a unified computational framework that reproduces these regimes and allows quantitative analysis of:
Jump-to-contact
Pull-off force
Effective Young’s modulus
Dissipated energy

🔷 2. Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

🔷 3. Define Physical Interaction Models
3.1 Lennard–Jones Force

In [2]:
def lennard_jones_force(r, epsilon, sigma):
    r = np.maximum(r, 1e-12)
    return 24 * epsilon * (2*(sigma**12)/r**13 - (sigma**6)/r**7)

3.2 Hertz Contact Force

In [3]:
def hertz_force(delta, E_star, R):
    delta = np.maximum(delta, 0.0)
    return (4.0/3.0) * E_star * np.sqrt(R) * delta**1.5

3.3 Adhesion Offset (DMT / JKR-inspired)

In [4]:
def adhesive_offset(R, Wadh, model="DMT"):
    if model.upper() == "DMT":
        C = 2.0
    elif model.upper() == "JKR":
        C = 1.5
    else:
        raise ValueError("model must be 'DMT' or 'JKR'")
    return -C * np.pi * R * Wadh

🔷 4. Total Interaction Force
This combines:
LJ (non-contact)
Hertz + adhesion (contact)
Optional hysteresis term during retraction

In [5]:
def total_interaction_force(D, params, branch="approach"):
    epsilon = params["epsilon"]
    sigma   = params["sigma"]
    E_star  = params["E_star"]
    R       = params["R"]
    Wadh    = params["Wadh"]
    adh_model = params["adh_model"]
    hyst_strength = params["hyst_strength"]
    r0 = params["r0"]

    r = D + r0
    F_LJ = lennard_jones_force(r, epsilon, sigma)

    delta = -D
    F_H = hertz_force(delta, E_star, R)
    F_adh = adhesive_offset(R, Wadh, model=adh_model)

    if branch == "retract":
        F_hyst = -hyst_strength * np.tanh(delta / params["hyst_delta0"])
    else:
        F_hyst = 0.0

    if D >= 0:
        return F_LJ
    else:
        return F_H + F_adh + F_hyst

🔷 5. Coupled Cantilever Solver
We solve quasi-static equilibrium:
k
x
=
F
i
n
t
e
r
a
c
t
i
o
n
(
D
)
with
D
=
z
p
i
e
z
o
−
x
kx=F
interaction
​
 (D)withD=z
piezo
​
 −x

In [6]:
def simulate_fd_curve(params, z_min=-30e-9, z_max=50e-9, n=2000):

    k = params["k"]

    z_approach = np.linspace(z_max, z_min, n)
    z_retract  = np.linspace(z_min, z_max, n)

    def solve_branch(z_array, branch_name, x0):
        x_prev = x0
        D_out, F_out = [], []

        for z in z_array:
            x = x_prev
            for _ in range(params["max_iter"]):
                D = z - x
                F_int = total_interaction_force(D, params, branch=branch_name)
                x = (1 - params["relax"]) * x + params["relax"] * (F_int / k)

            D = z - x
            F = k * x

            D_out.append(D)
            F_out.append(F)
            x_prev = x

        return np.array(D_out), np.array(F_out), x_prev

    D_app, F_app, x_last = solve_branch(z_approach, "approach", 0.0)
    D_ret, F_ret, _      = solve_branch(z_retract, "retract", x_last)

    return (D_app, F_app), (D_ret, F_ret)

🔷 6. Define Baseline Parameters

In [7]:
params = dict(
    k=0.2,                    # N/m
    epsilon=2e-20,            # J
    sigma=0.35e-9,            # m
    r0=0.35e-9,
    E_star=1e6,               # Pa
    R=20e-9,                  # m
    Wadh=50e-3,               # J/m^2
    adh_model="DMT",
    hyst_strength=2e-9,       # N
    hyst_delta0=2e-9,
    max_iter=80,
    relax=0.25
)

🔷 7. Interactive Force–Distance Simulation

Use the sliders below to explore how each physical parameter affects the force–distance curve in real time. The plot updates automatically and displays the pull-off force and dissipated energy.

In [ ]:
# ── Interactive sliders ──────────────────────────────────────────────
style  = {"description_width": "160px"}
layout = widgets.Layout(width="480px")

slider_k = widgets.FloatLogSlider(
    value=0.2, base=10, min=-2, max=1, step=0.1,
    description="Cantilever k (N/m)", style=style, layout=layout,
    readout_format=".3f"
)
slider_E = widgets.FloatLogSlider(
    value=1e6, base=10, min=5, max=10, step=0.25,
    description="E* (Pa)", style=style, layout=layout,
    readout_format=".2e"
)
slider_R = widgets.FloatSlider(
    value=20, min=5, max=100, step=5,
    description="Tip radius R (nm)", style=style, layout=layout
)
slider_Wadh = widgets.FloatSlider(
    value=50, min=0, max=200, step=5,
    description="W_adh (mJ/m²)", style=style, layout=layout
)
slider_hyst = widgets.FloatSlider(
    value=2.0, min=0, max=10, step=0.5,
    description="Hysteresis (nN)", style=style, layout=layout
)
dropdown_model = widgets.Dropdown(
    options=["DMT", "JKR"],
    value="DMT",
    description="Adhesion model", style=style, layout=layout
)

output_fd = widgets.Output()

def update_fd_curve(change=None):
    # Build params dict from slider values
    p = dict(
        k=slider_k.value,
        epsilon=2e-20,
        sigma=0.35e-9,
        r0=0.35e-9,
        E_star=slider_E.value,
        R=slider_R.value * 1e-9,
        Wadh=slider_Wadh.value * 1e-3,
        adh_model=dropdown_model.value,
        hyst_strength=slider_hyst.value * 1e-9,
        hyst_delta0=2e-9,
        max_iter=80,
        relax=0.25,
    )

    (D_a, F_a), (D_r, F_r) = simulate_fd_curve(p)

    pull_off = np.min(F_r) * 1e9
    E_app = np.trapezoid(F_a, D_a)
    E_ret = np.trapezoid(F_r, D_r)
    E_diss = abs(E_ret - E_app)

    with output_fd:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.plot(D_a * 1e9, F_a * 1e9, color="#2980B9", lw=2, label="Approach")
        ax.plot(D_r * 1e9, F_r * 1e9, color="#E74C3C", lw=2, label="Retraction")
        ax.axhline(0, color="grey", lw=0.5)
        ax.set_xlabel("Tip–sample separation (nm)")
        ax.set_ylabel("Force (nN)")
        ax.set_title("Simulated AFM Force–Distance Curve")
        ax.legend(loc="upper left")

        info = (f"Pull-off: {pull_off:.2f} nN  |  "
                f"E_diss: {E_diss:.2e} J  |  "
                f"Model: {dropdown_model.value}")
        ax.text(0.98, 0.02, info, transform=ax.transAxes,
                fontsize=9, ha="right", va="bottom",
                bbox=dict(facecolor="white", edgecolor="#ccc", alpha=0.85))
        plt.tight_layout()
        plt.show()

# Connect all widgets to the update function
for w in [slider_k, slider_E, slider_R, slider_Wadh, slider_hyst, dropdown_model]:
    w.observe(update_fd_curve, names="value")

controls = widgets.VBox([
    widgets.HBox([slider_k, slider_E]),
    widgets.HBox([slider_R, slider_Wadh]),
    widgets.HBox([slider_hyst, dropdown_model]),
])

display(controls, output_fd)
update_fd_curve()  # initial plot

In [ ]:
# ── Baseline run with default parameters (used by analysis cells below) ──
(D_app, F_app), (D_ret, F_ret) = simulate_fd_curve(params)

plt.figure(figsize=(8, 4.5))
plt.plot(D_app*1e9, F_app*1e9, color="#2980B9", lw=2, label="Approach")
plt.plot(D_ret*1e9, F_ret*1e9, color="#E74C3C", lw=2, label="Retraction")
plt.axhline(0, color="grey", lw=0.5)
plt.xlabel("Tip–sample separation (nm)")
plt.ylabel("Force (nN)")
plt.title("Simulated AFM Force–Distance Curve (baseline parameters)")
plt.legend()
plt.tight_layout()
plt.show()

🔷 8. Extract Pull-Off Force and Dissipated Energy

In [9]:
pull_off = np.min(F_ret) * 1e9

E_app = np.trapezoid(F_app, D_app)
E_ret = np.trapezoid(F_ret, D_ret)
E_diss = abs(E_ret - E_app)

print(f"Pull-off force: {pull_off:.3f} nN")
print(f"Dissipated energy: {E_diss:.3e} J")

Pull-off force: -6.100 nN
Dissipated energy: 3.150e-17 J


🔷 9. Hertz Fit to Extract Young’s Modulus

In [10]:
def fit_hertz(D, F, R, Wadh, adh_model="DMT"):
    # Select the contact region (D < 0 means indentation)
    mask = D < 0
    delta = -D[mask]
    F_contact = F[mask]

    # Subtract adhesion offset so we fit pure Hertz (elastic) part
    F_adh = adhesive_offset(R, Wadh, model=adh_model)
    F_corrected = F_contact - F_adh

    # Only fit the repulsive regime with sufficient indentation
    valid = (delta > 0.5e-9) & (F_corrected > 0)
    delta = delta[valid]
    F_corrected = F_corrected[valid]

    if len(delta) == 0:
        print("Warning: no valid contact data for Hertz fit")
        return np.nan

    X = delta**1.5
    Y = F_corrected

    A = np.sum(X * Y) / np.sum(X * X)
    E_star_fit = (3/4) * A / np.sqrt(R)
    return E_star_fit

E_fit = fit_hertz(D_app, F_app, params["R"], params["Wadh"], params["adh_model"])

print(f"Input E*: {params['E_star']:.2e} Pa")
print(f"Fitted E*: {E_fit:.2e} Pa")

Input E*: 1.00e+06 Pa
Fitted E*: 4.41e+07 Pa


🔷 10. Interactive Parameter Exploration

Compare how different values of a chosen parameter affect the approach curve. Select a parameter and adjust its range to see overlaid curves.

In [ ]:
# ── Interactive parameter comparison ─────────────────────────────────
param_select = widgets.Dropdown(
    options=[
        ("Cantilever k (N/m)", "k"),
        ("Tip radius R (nm)", "R"),
        ("Work of adhesion W_adh (mJ/m²)", "Wadh"),
        ("Reduced modulus E* (Pa)", "E_star"),
        ("Hysteresis strength (nN)", "hyst_strength"),
    ],
    value="k",
    description="Sweep parameter",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="420px"),
)

# Default sweep ranges for each parameter (3 values each, in display units)
sweep_defaults = {
    "k":             {"values": [0.05, 0.2, 1.0],        "unit": "N/m",   "scale": 1},
    "R":             {"values": [10, 20, 50],             "unit": "nm",    "scale": 1e-9},
    "Wadh":          {"values": [10, 50, 150],            "unit": "mJ/m²", "scale": 1e-3},
    "E_star":        {"values": [1e5, 1e6, 1e7],          "unit": "Pa",    "scale": 1},
    "hyst_strength": {"values": [0.5, 2.0, 8.0],          "unit": "nN",    "scale": 1e-9},
}

sweep_v1 = widgets.FloatText(description="Value 1", style={"description_width": "60px"}, layout=widgets.Layout(width="180px"))
sweep_v2 = widgets.FloatText(description="Value 2", style={"description_width": "60px"}, layout=widgets.Layout(width="180px"))
sweep_v3 = widgets.FloatText(description="Value 3", style={"description_width": "60px"}, layout=widgets.Layout(width="180px"))

def set_defaults(change=None):
    d = sweep_defaults[param_select.value]["values"]
    sweep_v1.value, sweep_v2.value, sweep_v3.value = d[0], d[1], d[2]

param_select.observe(set_defaults, names="value")
set_defaults()

output_explore = widgets.Output()

def update_exploration(btn=None):
    key = param_select.value
    info = sweep_defaults[key]
    scale = info["scale"]
    unit  = info["unit"]
    values = [sweep_v1.value, sweep_v2.value, sweep_v3.value]

    base = dict(
        k=0.2, epsilon=2e-20, sigma=0.35e-9, r0=0.35e-9,
        E_star=1e6, R=20e-9, Wadh=50e-3, adh_model="DMT",
        hyst_strength=2e-9, hyst_delta0=2e-9, max_iter=80, relax=0.25,
    )

    colors = ["#2980B9", "#E74C3C", "#27AE60"]

    with output_explore:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

        for val, col in zip(values, colors):
            p = base.copy()
            p[key] = val * scale
            (D_a, F_a), (D_r, F_r) = simulate_fd_curve(p)

            label = f"{key}={val} {unit}"
            axes[0].plot(D_a * 1e9, F_a * 1e9, color=col, lw=2, label=label)
            axes[1].plot(D_a * 1e9, F_a * 1e9, color=col, lw=2, alpha=0.4)
            axes[1].plot(D_r * 1e9, F_r * 1e9, color=col, lw=2, ls="--", label=label)

        axes[0].axhline(0, color="grey", lw=0.5)
        axes[0].set_xlabel("Separation (nm)")
        axes[0].set_ylabel("Force (nN)")
        axes[0].set_title("Approach curves")
        axes[0].legend(fontsize=9)

        axes[1].axhline(0, color="grey", lw=0.5)
        axes[1].set_xlabel("Separation (nm)")
        axes[1].set_ylabel("Force (nN)")
        axes[1].set_title("Approach (solid) & Retraction (dashed)")
        axes[1].legend(fontsize=9)

        plt.tight_layout()
        plt.show()

run_btn = widgets.Button(description="Run comparison", button_style="primary",
                         layout=widgets.Layout(width="160px"))
run_btn.on_click(update_exploration)

display(
    widgets.HBox([param_select, run_btn]),
    widgets.HBox([sweep_v1, sweep_v2, sweep_v3]),
    output_explore,
)
update_exploration()

🔷 11. Discussion Questions
Why does jump-to-contact depend on cantilever stiffness?
Why does adhesion differ between DMT and JKR-inspired models?
Why does incorrect contact-point identification bias modulus extraction?
What physical processes could the hysteresis term represent?

🔷 12. Core Conceptual Message
A force–distance curve is the measurable outcome of a coupled system:
Atomic interaction + Surface thermodynamics + Contact mechanics + Cantilever dynamics.
Quantitative AFM requires identifying which regime dominates and which physical model applies.
Precision without conceptual awareness leads to systematic error.